# janus_01 – kiértékelés az eval halmazon

**Beállítás a jobb oldali panelen, Run All előtt:**
1. **Accelerator** → GPU T4 x2.
2. **Internet** → On.
3. **Input → Add Input** → `asvpoof-2019-dataset` (awsaf49).
4. **Input → Add Input → Your Work** → a tanító notebook (02_train_kaggle), hogy a `janus_best.pt` checkpoint elérhető legyen. Alternatíva: töltsd fel a `.pt` fájlt Kaggle-adathalmazként, és azt add hozzá.

Az eval halmaz 71 ezer fájl, a spektrogramok kiszámítása ~60 perc. Utána a pontozás maga percek.

Kimenet a `/kaggle/working/out` mappában: `eval_scores.csv` (minden fájl pontszáma) és `eval_per_attack.csv` (EER támadástípusonként).

## 0. Környezet ellenőrzése

In [ ]:
import os, sys, torch
from pathlib import Path

assert torch.cuda.is_available(), 'NINCS GPU. Accelerator → GPU T4 x2, majd Run All újra.'
print('GPU:', torch.cuda.get_device_name(0))
inputs = list(Path('/kaggle/input').iterdir())
print('/kaggle/input:', [p.name for p in inputs])
assert inputs, 'NINCS INPUT. Add hozzá az adathalmazt és a tanító notebook kimenetét.'

## 1. Kód

In [ ]:
REPO = 'https://github.com/Csaszi05/janus_01.git'
if not os.path.exists('/kaggle/working/janus_01'):
    !git clone -q {REPO} /kaggle/working/janus_01
else:
    !cd /kaggle/working/janus_01 && git pull -q
%cd /kaggle/working/janus_01
!git log --oneline | head -3
!pip install -q -r requirements.txt

## 2. Adat és checkpoint megkeresése

In [ ]:
def find_dir(root, name, depth=8):
    # szélességi keresés, max 8 szint mélyen; nem járja be a flac fájlokat
    level = [Path(root)]
    for _ in range(depth):
        nxt = []
        for d in level:
            for p in d.iterdir():
                if not p.is_dir():
                    continue
                if p.name == name:
                    return p
                nxt.append(p)
        level = nxt
    return None

proto = find_dir('/kaggle/input', 'ASVspoof2019_LA_cm_protocols')
assert proto, 'Nem találom az ASVspoof2019_LA_cm_protocols mappát a /kaggle/input alatt.'
DATA_ROOT = str(proto.parent); print('DATA_ROOT  =', DATA_ROOT)

ck_dir = find_dir('/kaggle/input', 'out')
cks = [p for p in Path('/kaggle/input').glob('*/*/*/janus_best.pt')] + ([ck_dir / 'janus_best.pt'] if ck_dir and (ck_dir / 'janus_best.pt').exists() else [])
assert cks, 'Nem találom a janus_best.pt fájlt. Add hozzá a tanító notebook kimenetét (Add Input → Your Work).'
CHECKPOINT = str(cks[0]); print('CHECKPOINT =', CHECKPOINT)

ck = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
print('checkpoint epoch:', ck['epoch'], '| audio config:', ck['config']['audio'])

## 3. Ellenőrzés a dev halmazon

A tanításkor mért legjobb dev EER-t (4.88%) kell visszakapnunk. Ha nem, valami nem stimmel a checkpointtal vagy a feature-számítással. Ez is cache-t épít (~20 perc), de a dev-bontás támadásonként önmagában is hasznos.

In [ ]:
!python src/evaluate.py --data_root {DATA_ROOT} --checkpoint {CHECKPOINT} --split dev --workers 2 --cache_dir /tmp/cache --out /kaggle/working/out

## 4. Eval halmaz

Itt vannak az ismeretlen támadások (A07–A19). Ez az igazi mérőszám.

In [ ]:
!python src/evaluate.py --data_root {DATA_ROOT} --checkpoint {CHECKPOINT} --split eval --workers 2 --cache_dir /tmp/cache --out /kaggle/working/out

## 5. Eredmény

In [ ]:
import pandas as pd
per = pd.read_csv('/kaggle/working/out/eval_per_attack.csv')
per['eer_%'] = (per.eer * 100).round(2)
print(per[['system', 'n', 'eer_%']].to_string(index=False))
!ls -la /kaggle/working/out